In [2]:
import pandas as pd
import psycopg2
from pymongo import MongoClient
from sqlalchemy import create_engine


In [3]:
mongo_client = MongoClient(f'mongodb://localhost:27017/')
db = mongo_client['oto_db']
collection = db['oto_collection']

In [115]:
# 2. Fetch Data from MongoDB
data = list(collection.find())
df = pd.DataFrame(data)

In [61]:
import re

In [116]:
def xuliTien(price):
    # Chuyển price thành chuỗi và xóa mọi khoảng trắng
    price = str(price).replace(" ", "").strip("[]'")  # Loại bỏ ký tự không cần thiết

    # Khởi tạo giá trị tổng
    total_value = 0

    # Kiểm tra và tính toán giá trị
    if 'Tỷ' in price:
        # Tách theo 'Tỷ' và xử lý phần trước
        parts = price.split('Tỷ')
        if parts[0]:  # Kiểm tra nếu có giá trị trước 'Tỷ'
            total_value += int(parts[0]) * 1000  # Chuyển Tỷ thành số

        # Kiểm tra nếu có giá trị triệu sau 'Tỷ'
        if len(parts) > 1:
            parts[1] = parts[1].replace("Triệu", "")  # Làm sạch phần triệu
            if parts[1]:  # Đảm bảo không rỗng
                total_value += int(parts[1]) * 1  # Chuyển Triệu thành số

    elif 'Triệu' in price:
        # Nếu chỉ có triệu
        parts = price.split('Triệu')
        if parts[0]:  # Kiểm tra nếu có giá trị trước 'Triệu'
            total_value += int(parts[0]) * 1  # Chuyển Triệu thành số

    return total_value

# Áp dụng hàm cho cột price trong dataframe
df['price'] = df['price'].apply(xuliTien)

# In ra dataframe đã chỉnh sửa
print(df[['price']])


      price
0       255
1      1235
2      2850
3       775
4      3200
...     ...
2072    690
2073    650
2074    255
2075   1560
2076    315

[2077 rows x 1 columns]


In [104]:
def xuliTien(price):
    # Đảm bảo price là chuỗi và xóa mọi khoảng trắng
    price = str(price).replace(" ", "").strip()

    # Khởi tạo giá trị tổng
    total_value = 0

    # Kiểm tra sự hiện diện của "tỷ" và "triệu" trong chuỗi
    if 'tỷ' in price:
        # Tách theo 'tỷ' và xử lý phần đầu tiên
        parts = price.split('tỷ')
        if parts[0]:  # Kiểm tra nếu có giá trị trước 'tỷ'
            total_value += float(parts[0]) * 1000000000  # Chuyển tỷ thành số

        # Kiểm tra nếu có giá trị sau 'tỷ' cho 'triệu'
        if len(parts) > 1:
            parts[1] = parts[1].replace("triệu", "").strip()  # Làm sạch phần thứ hai
            if parts[1]:  # Đảm bảo không rỗng
                total_value += float(parts[1]) * 1000000  # Chuyển triệu thành số

    elif 'triệu' in price:
        # Nếu chỉ có giá trị triệu
        parts = price.split('triệu')
        if parts[0]:  # Kiểm tra nếu có giá trị trước 'triệu'
            total_value += float(parts[0]) * 1000000  # Chuyển triệu thành số

    return total_value

# Áp dụng hàm cho cột price trong dataframe
df['price'] = df['price'].apply(xuliTien)

# In ra dataframe đã chỉnh sửa
print(df[['price']])


      price
0         0
1         0
2         0
3         0
4         0
...     ...
2072      0
2073      0
2074      0
2075      0
2076      0

[2077 rows x 1 columns]


In [ ]:
data['price'] = data['price'].str.replace(' Triệu', '')

In [103]:

df['price']

0          255 
1       1  235 
2       2  850 
3          775 
4       3  200 
         ...   
2072       690 
2073       650 
2074       255 
2075    1  560 
2076       315 
Name: price, Length: 2077, dtype: object

In [91]:
df.dtypes

_id            object
phonenumber    object
sellername     object
address        object
status         object
year           object
code           object
carname        object
price          object
city           object
parameter      object
content        object
dtype: object

In [147]:
data = list(collection.find({}))
df = pd.DataFrame(data)

In [148]:
df[['address','content','parameter']]

,address,content,parameter
0,"[ Khám Lạng, Lục Nam Bắc Giang ]",[[ Innova G sx 2011 tư nhân máy số zin k đâm đ...,"[ *Xe lắp ráp trong nước, màu bạc, máy xăng 2...."
1,"[ Số 1 Nguyễn Văn Huyên, Cầu Giấy Hà Nội ]",[[ - Chào bán Vinfast vf9\n- SUV ngoại cỡ với ...,"[ *Xe lắp ráp trong nước, màu đen, xe điện , s..."
2,"[ 100 Nguyễn Văn Cừ, Long Biên Hà Nội ]",[[ HÀNG MỚI VỀ !!!\nGIA BẢO AUTO 100 NVC-LB-HN...,"[ *Xe nhập khẩu, màu trắng, máy xăng 3.0 L, số..."
3,"[ Vinhomes Smart City Tây Mỗ, Nam Từ Liêm Hà N...",[[ Mercedes C300 AMG sản xuất 2016 đen nội thấ...,"[ *Xe lắp ráp trong nước, màu đen, máy xăng 2...."
4,"[ 136 Phạm Văn Đồng, P. Xuân Đỉnh, Q. Bắc Từ L...","[[ Porsche macan 2022 xe màu đen nt kem, xe ch...","[ *Xe nhập khẩu, màu đen, máy xăng 2.0 L, số t..."
...,...,...,...
1015,"[ 79 Nguyễn Chánh, Trung Hoà, Cầu Giấy Hà Nội ]","[[ Biển HN, 6,3vạn\nBao test, check hãng\nGiá ...","[ *Xe lắp ráp trong nước, màu đen, máy xăng 1...."
1016,"[ 387 QL. 13, Phường Hiệp Bình Phước, Quận Thủ...",[[ Hyundai Grand i10 1.2AT 2023\n⚙️Số tự động ...,"[ *Xe lắp ráp trong nước, màu trắng, máy xăng ..."
1017,"[ 210 Võ Chí Công,Xuân La,Tây Hồ Hà Nội ]",[[ Mới về Mercedes Benz GLC250 4Matic 2018 bản...,"[ *Xe lắp ráp trong nước, màu trắng, máy xăng ..."
1018,"[ 79 Nguyễn Chánh, Trung Hoà, Cầu Giấy Hà Nội ]","[[ Biển HN, 10vạn, tên công ty ko xuất được VA...","[ *Xe nhập khẩu, màu bạc, máy xăng 1.5 L, số t..."


In [149]:
import re

# Hàm xử lý văn bản: xóa ký tự đặc biệt và dấu ngoặc
def clean_text(text):
    if isinstance(text, str):  # Kiểm tra xem text có phải là chuỗi không
        text = text.replace("*", "")  # Xóa dấu *
        text = re.sub(r'[^\w\sÀ-ỹ]', '', text)  # Xóa ký tự đặc biệt, giữ lại chữ có dấu và khoảng trắng
        return text.strip()  # Xóa khoảng trắng đầu và cuối
    return text  # Trả về giá trị ban đầu nếu không phải chuỗi

# Áp dụng hàm clean_text cho các cột 'address', 'content', và 'parameter'
df['address'] = df['address'].apply(clean_text)
df['content'] = df['content'].apply(clean_text)
df['parameter'] = df['parameter'].apply(clean_text)




In [150]:
df[['address','content','parameter']]

,address,content,parameter
0,"[ Khám Lạng, Lục Nam Bắc Giang ]",[[ Innova G sx 2011 tư nhân máy số zin k đâm đ...,"[ *Xe lắp ráp trong nước, màu bạc, máy xăng 2...."
1,"[ Số 1 Nguyễn Văn Huyên, Cầu Giấy Hà Nội ]",[[ - Chào bán Vinfast vf9\n- SUV ngoại cỡ với ...,"[ *Xe lắp ráp trong nước, màu đen, xe điện , s..."
2,"[ 100 Nguyễn Văn Cừ, Long Biên Hà Nội ]",[[ HÀNG MỚI VỀ !!!\nGIA BẢO AUTO 100 NVC-LB-HN...,"[ *Xe nhập khẩu, màu trắng, máy xăng 3.0 L, số..."
3,"[ Vinhomes Smart City Tây Mỗ, Nam Từ Liêm Hà N...",[[ Mercedes C300 AMG sản xuất 2016 đen nội thấ...,"[ *Xe lắp ráp trong nước, màu đen, máy xăng 2...."
4,"[ 136 Phạm Văn Đồng, P. Xuân Đỉnh, Q. Bắc Từ L...","[[ Porsche macan 2022 xe màu đen nt kem, xe ch...","[ *Xe nhập khẩu, màu đen, máy xăng 2.0 L, số t..."
...,...,...,...
1015,"[ 79 Nguyễn Chánh, Trung Hoà, Cầu Giấy Hà Nội ]","[[ Biển HN, 6,3vạn\nBao test, check hãng\nGiá ...","[ *Xe lắp ráp trong nước, màu đen, máy xăng 1...."
1016,"[ 387 QL. 13, Phường Hiệp Bình Phước, Quận Thủ...",[[ Hyundai Grand i10 1.2AT 2023\n⚙️Số tự động ...,"[ *Xe lắp ráp trong nước, màu trắng, máy xăng ..."
1017,"[ 210 Võ Chí Công,Xuân La,Tây Hồ Hà Nội ]",[[ Mới về Mercedes Benz GLC250 4Matic 2018 bản...,"[ *Xe lắp ráp trong nước, màu trắng, máy xăng ..."
1018,"[ 79 Nguyễn Chánh, Trung Hoà, Cầu Giấy Hà Nội ]","[[ Biển HN, 10vạn, tên công ty ko xuất được VA...","[ *Xe nhập khẩu, màu bạc, máy xăng 1.5 L, số t..."
